# CoNSeP 数据集处理
本 notebook 将 consep_raw 中的实例分割标注转换为 Faster R-CNN 可用的 CSV 检测标注。
处理逻辑与 MCSpatNet 的 CoNSeP 三类映射保持一致：1 = inflammatory，2 = epithelial（3+4），3 = stromal / spindle（1+5+6+7）。
不做 patch 裁剪，直接以原始整张图像为单位导出 images、annotations/boxes.csv 和 metadata/classes.csv。
Train 目录会按照 data_splits/consep/train_split.txt 与 val_split.txt 划分为 train / val，Test 目录直接作为 test。

In [9]:
from __future__ import annotations

import csv
import shutil
from pathlib import Path

import numpy as np
from PIL import Image
from scipy.io import loadmat
from tqdm.auto import tqdm

SOURCE_ROOT = Path("consep_raw")
OUTPUT_ROOT = Path("data") / "CoNSeP"
TRAIN_SPLIT_PATH = Path("..") / "data_splits" / "consep" / "train_split.txt"
VAL_SPLIT_PATH = Path("..") / "data_splits" / "consep" / "val_split.txt"
OVERWRITE_OUTPUT = False

CLASS_GROUP_MAPPING_DICT = {1: [2], 2: [3, 4], 3: [1, 5, 6, 7]}
CLASS_NAMES = {
    1: "inflammatory",
    2: "epithelial",
    3: "stromal",
}

LABEL_TO_GROUP = {
    source_label: group_label
    for group_label, source_labels in CLASS_GROUP_MAPPING_DICT.items()
    for source_label in source_labels
}

In [2]:
def read_split_stems(split_path: Path) -> set[str]:
    stems: set[str] = set()
    with split_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            name = line.strip()
            if not name:
                continue
            stems.add(Path(name).stem)
    return stems


def ensure_output_root(output_root: Path, overwrite: bool) -> None:
    if output_root.exists() and overwrite:
        shutil.rmtree(output_root)

    (output_root / "images" / "train").mkdir(parents=True, exist_ok=True)
    (output_root / "images" / "val").mkdir(parents=True, exist_ok=True)
    (output_root / "images" / "test").mkdir(parents=True, exist_ok=True)
    (output_root / "annotations").mkdir(parents=True, exist_ok=True)
    (output_root / "metadata").mkdir(parents=True, exist_ok=True)


def resolve_train_split(sample_stem: str, train_stems: set[str], val_stems: set[str]) -> str:
    if sample_stem in train_stems:
        return "train"
    if sample_stem in val_stems:
        return "val"
    raise ValueError(f"{sample_stem} 不在 train/val split 文件中")


def extract_boxes_from_mat(mat_path: Path) -> list[dict[str, int | str]]:
    mat = loadmat(mat_path)
    inst_map = mat["inst_map"]
    inst_type = np.asarray(mat["inst_type"]).reshape(-1)

    boxes: list[dict[str, int | str]] = []
    instance_ids = np.unique(inst_map)
    instance_ids = instance_ids[instance_ids > 0]

    for instance_id in instance_ids:
        instance_index = int(instance_id) - 1
        if instance_index < 0 or instance_index >= len(inst_type):
            continue

        raw_label = int(inst_type[instance_index])
        group_label = LABEL_TO_GROUP.get(raw_label)
        if group_label is None:
            continue

        ys, xs = np.where(inst_map == instance_id)
        if ys.size == 0 or xs.size == 0:
            continue

        xmin = int(xs.min())
        ymin = int(ys.min())
        xmax = int(xs.max()) + 1
        ymax = int(ys.max()) + 1

        if xmax <= xmin or ymax <= ymin:
            continue

        boxes.append(
            {
                "xmin": xmin,
                "ymin": ymin,
                "xmax": xmax,
                "ymax": ymax,
                "label": CLASS_NAMES[group_label],
                "label_id": group_label,
            }
        )

    return boxes

In [3]:
def export_split(
    image_dir: Path,
    label_dir: Path,
    split_name: str,
    output_root: Path,
    rows: list[dict[str, int | str]],
    known_stems: set[str] | None = None,
 ) -> dict[str, int]:
    image_paths = sorted(image_dir.glob("*.png"))
    image_paths += sorted(image_dir.glob("*.jpg"))
    image_paths += sorted(image_dir.glob("*.jpeg"))
    image_paths += sorted(image_dir.glob("*.tif"))
    image_paths += sorted(image_dir.glob("*.tiff"))

    image_count = 0
    box_count = 0

    for image_path in tqdm(image_paths, desc=f"导出 {split_name}", unit="image"):
        sample_stem = image_path.stem
        if known_stems is not None and sample_stem not in known_stems:
            continue

        mat_path = label_dir / f"{sample_stem}.mat"
        if not mat_path.exists():
            raise FileNotFoundError(f"找不到与图像对应的标注文件: {mat_path}")

        destination = output_root / "images" / split_name / image_path.name
        if destination.resolve() != image_path.resolve():
            shutil.copy2(image_path, destination)

        relative_image_path = Path("images") / split_name / image_path.name
        boxes = extract_boxes_from_mat(mat_path)
        image_count += 1

        if not boxes:
            rows.append(
                {
                    "split": split_name,
                    "image_path": relative_image_path.as_posix(),
                    "xmin": "",
                    "ymin": "",
                    "xmax": "",
                    "ymax": "",
                    "label": "background",
                    "label_id": 0,
                    "is_negative": 1,
                }
            )
            continue

        for box in boxes:
            rows.append(
                {
                    "split": split_name,
                    "image_path": relative_image_path.as_posix(),
                    "xmin": box["xmin"],
                    "ymin": box["ymin"],
                    "xmax": box["xmax"],
                    "ymax": box["ymax"],
                    "label": box["label"],
                    "label_id": box["label_id"],
                    "is_negative": 0,
                }
            )
            box_count += 1

    return {"images": image_count, "boxes": box_count}

In [4]:
def prepare_consep_detection_dataset(
    source_root: Path = SOURCE_ROOT,
    output_root: Path = OUTPUT_ROOT,
    train_split_path: Path = TRAIN_SPLIT_PATH,
    val_split_path: Path = VAL_SPLIT_PATH,
    overwrite: bool = OVERWRITE_OUTPUT,
 ) -> None:
    source_root = Path(source_root)
    output_root = Path(output_root)
    train_split_path = Path(train_split_path)
    val_split_path = Path(val_split_path)

    ensure_output_root(output_root, overwrite)

    train_stems = read_split_stems(train_split_path)
    val_stems = read_split_stems(val_split_path)
    overlap = train_stems & val_stems
    if overlap:
        raise ValueError(f"train/val split 存在重复样本: {sorted(overlap)[:5]}")

    rows: list[dict[str, int | str]] = []

    train_dir = source_root / "Train"
    test_dir = source_root / "Test"

    train_images_dir = train_dir / "Images"
    train_labels_dir = train_dir / "Labels"
    test_images_dir = test_dir / "Images"
    test_labels_dir = test_dir / "Labels"

    summary_train = export_split(
        train_images_dir,
        train_labels_dir,
        "train",
        output_root,
        rows,
        known_stems=train_stems,
    )
    summary_val = export_split(
        train_images_dir,
        train_labels_dir,
        "val",
        output_root,
        rows,
        known_stems=val_stems,
    )
    summary_test = export_split(
        test_images_dir,
        test_labels_dir,
        "test",
        output_root,
        rows,
    )

    fieldnames = [
        "split",
        "image_path",
        "xmin",
        "ymin",
        "xmax",
        "ymax",
        "label",
        "label_id",
        "is_negative",
    ]

    with (output_root / "annotations" / "boxes.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    with (output_root / "metadata" / "classes.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["label_id", "label"])
        writer.writeheader()
        writer.writerow({"label_id": 0, "label": "background"})
        for label_id, label_name in CLASS_NAMES.items():
            writer.writerow({"label_id": label_id, "label": label_name})

    print(f"输出目录: {output_root.resolve()}")
    print(
        "split 统计:",
        {
            "train": summary_train,
            "val": summary_val,
            "test": summary_test,
        },
    )
    print(f"总标注框数: {sum(int(row['is_negative']) == 0 for row in rows)}")

In [5]:
# 按需修改上面的参数后，再运行本单元。

prepare_consep_detection_dataset(
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_ROOT,
    train_split_path=TRAIN_SPLIT_PATH,
    val_split_path=VAL_SPLIT_PATH,
    overwrite=OVERWRITE_OUTPUT,
)

导出 test: 100%|██████████| 14/14 [00:25<00:00,  1.83s/image]

输出目录: E:\WYH\CV\MCSpatNet\contrast\data\CoNSeP
split 统计: {'train': {'images': 22, 'boxes': 13040}, 'val': {'images': 5, 'boxes': 2515}, 'test': {'images': 14, 'boxes': 8777}}
总标注框数: 24332


## 另一种处理逻辑：固定切分 4 个 500x500 patch
这一组单元与上面的整图导出逻辑保持一致，唯一差异是：
- 每张 1000x1000 图像固定切成 4 个不重叠的 500x500 patch
- patch 文件名中显式记录来源原图和左上角坐标
- 先按原图严格划分 train / val / test，再生成 patch，因此同一原图的 patch 不会跨集合

In [7]:
PATCH_OUTPUT_ROOT = Path("data") / "consep_patch"
PATCH_SIZE = 500
SOURCE_IMAGE_SIZE = 1000


def _iter_fixed_patch_specs(image_size: int = SOURCE_IMAGE_SIZE, patch_size: int = PATCH_SIZE):
    if image_size % patch_size != 0:
        raise ValueError("image_size 必须能被 patch_size 整除")
    for patch_y in range(0, image_size, patch_size):
        for patch_x in range(0, image_size, patch_size):
            yield patch_x, patch_y


def _box_center_in_patch(box: dict[str, int | str], patch_x: int, patch_y: int, patch_size: int) -> bool:
    center_x = (int(box["xmin"]) + int(box["xmax"])) / 2.0
    center_y = (int(box["ymin"]) + int(box["ymax"])) / 2.0
    return patch_x <= center_x < patch_x + patch_size and patch_y <= center_y < patch_y + patch_size


def _clip_box_to_patch(box: dict[str, int | str], patch_x: int, patch_y: int, patch_size: int):
    xmin = max(0, int(box["xmin"]) - patch_x)
    ymin = max(0, int(box["ymin"]) - patch_y)
    xmax = min(patch_size, int(box["xmax"]) - patch_x)
    ymax = min(patch_size, int(box["ymax"]) - patch_y)
    if xmax <= xmin or ymax <= ymin:
        return None
    return {
        "xmin": xmin,
        "ymin": ymin,
        "xmax": xmax,
        "ymax": ymax,
        "label": str(box["label"]),
        "label_id": int(box["label_id"]),
    }


def export_split_as_fixed_patches(
    image_dir: Path,
    label_dir: Path,
    split_name: str,
    output_root: Path,
    rows: list[dict[str, int | str]],
    known_stems: set[str] | None = None,
    image_size: int = SOURCE_IMAGE_SIZE,
    patch_size: int = PATCH_SIZE,
 ) -> dict[str, int]:
    image_paths = sorted(image_dir.glob("*.png"))
    image_count = 0
    patch_count = 0
    box_count = 0

    for image_path in tqdm(image_paths, desc=f"切分 {split_name} patch", unit="image"):
        sample_stem = image_path.stem
        if known_stems is not None and sample_stem not in known_stems:
            continue

        mat_path = label_dir / f"{sample_stem}.mat"
        if not mat_path.exists():
            raise FileNotFoundError(f"找不到与图像对应的标注文件: {mat_path}")

        image = np.array(Image.open(image_path).convert("RGB"))
        if image.shape[0] != image_size or image.shape[1] != image_size:
            raise ValueError(
                f"{image_path.name} 尺寸为 {image.shape[1]}x{image.shape[0]}，不满足 {image_size}x{image_size}"
            )

        full_boxes = extract_boxes_from_mat(mat_path)
        image_count += 1

        for patch_x, patch_y in _iter_fixed_patch_specs(image_size=image_size, patch_size=patch_size):
            patch = image[patch_y:patch_y + patch_size, patch_x:patch_x + patch_size]
            patch_name = f"{sample_stem}__srcx{patch_x}_srcy{patch_y}.png"
            destination = output_root / "images" / split_name / patch_name
            Image.fromarray(patch).save(destination)

            relative_image_path = Path("images") / split_name / patch_name
            patch_boxes: list[dict[str, int | str]] = []
            for box in full_boxes:
                if not _box_center_in_patch(box, patch_x, patch_y, patch_size):
                    continue
                clipped_box = _clip_box_to_patch(box, patch_x, patch_y, patch_size)
                if clipped_box is not None:
                    patch_boxes.append(clipped_box)

            patch_count += 1
            if not patch_boxes:
                rows.append(
                    {
                        "split": split_name,
                        "image_path": relative_image_path.as_posix(),
                        "xmin": "",
                        "ymin": "",
                        "xmax": "",
                        "ymax": "",
                        "label": "background",
                        "label_id": 0,
                        "is_negative": 1,
                    }
                )
                continue

            for box in patch_boxes:
                rows.append(
                    {
                        "split": split_name,
                        "image_path": relative_image_path.as_posix(),
                        "xmin": box["xmin"],
                        "ymin": box["ymin"],
                        "xmax": box["xmax"],
                        "ymax": box["ymax"],
                        "label": box["label"],
                        "label_id": box["label_id"],
                        "is_negative": 0,
                    }
                )
                box_count += 1

    return {"images": image_count, "patches": patch_count, "boxes": box_count}


def prepare_consep_patch_detection_dataset(
    source_root: Path = SOURCE_ROOT,
    output_root: Path = PATCH_OUTPUT_ROOT,
    train_split_path: Path = TRAIN_SPLIT_PATH,
    val_split_path: Path = VAL_SPLIT_PATH,
    overwrite: bool = OVERWRITE_OUTPUT,
    image_size: int = SOURCE_IMAGE_SIZE,
    patch_size: int = PATCH_SIZE,
 ) -> None:
    source_root = Path(source_root)
    output_root = Path(output_root)
    train_split_path = Path(train_split_path)
    val_split_path = Path(val_split_path)

    ensure_output_root(output_root, overwrite)

    train_stems = read_split_stems(train_split_path)
    val_stems = read_split_stems(val_split_path)
    overlap = train_stems & val_stems
    if overlap:
        raise ValueError(f"train/val split 存在重复样本: {sorted(overlap)[:5]}")

    rows: list[dict[str, int | str]] = []
    train_dir = source_root / "Train"
    test_dir = source_root / "Test"

    summary_train = export_split_as_fixed_patches(
        train_dir / "Images",
        train_dir / "Labels",
        "train",
        output_root,
        rows,
        known_stems=train_stems,
        image_size=image_size,
        patch_size=patch_size,
    )
    summary_val = export_split_as_fixed_patches(
        train_dir / "Images",
        train_dir / "Labels",
        "val",
        output_root,
        rows,
        known_stems=val_stems,
        image_size=image_size,
        patch_size=patch_size,
    )
    summary_test = export_split_as_fixed_patches(
        test_dir / "Images",
        test_dir / "Labels",
        "test",
        output_root,
        rows,
        image_size=image_size,
        patch_size=patch_size,
    )

    fieldnames = [
        "split",
        "image_path",
        "xmin",
        "ymin",
        "xmax",
        "ymax",
        "label",
        "label_id",
        "is_negative",
    ]
    with (output_root / "annotations" / "boxes.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    with (output_root / "metadata" / "classes.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["label_id", "label"])
        writer.writeheader()
        writer.writerow({"label_id": 0, "label": "background"})
        for label_id, label_name in CLASS_NAMES.items():
            writer.writerow({"label_id": label_id, "label": label_name})

    print(f"patch 输出目录: {output_root.resolve()}")
    print(
        "严格 split 统计:",
        {
            "train": summary_train,
            "val": summary_val,
            "test": summary_test,
        },
    )
    print(f"总 patch 标注框数: {sum(int(row['is_negative']) == 0 for row in rows)}")

In [8]:
# 运行这一组单元后，输出会写入 data/consep_patch。

prepare_consep_patch_detection_dataset(
    source_root=SOURCE_ROOT,
    output_root=PATCH_OUTPUT_ROOT,
    train_split_path=TRAIN_SPLIT_PATH,
    val_split_path=VAL_SPLIT_PATH,
    overwrite=OVERWRITE_OUTPUT,
    image_size=SOURCE_IMAGE_SIZE,
    patch_size=PATCH_SIZE,
)

切分 test patch: 100%|██████████| 14/14 [00:31<00:00,  2.24s/image]

patch 输出目录: E:\WYH\CV\MCSpatNet\contrast\data\consep_patch
严格 split 统计: {'train': {'images': 22, 'patches': 88, 'boxes': 13040}, 'val': {'images': 5, 'patches': 20, 'boxes': 2515}, 'test': {'images': 14, 'patches': 56, 'boxes': 8777}}
总 patch 标注框数: 24332


## CoNSeP_point 目录整理与五折划分
这一部分把 `data/CoNSeP_point/CoNSeP_train` 和 `data/CoNSeP_point/CoNSeP_test` 整理成统一目录结构：
- `data/CoNSeP_point/images`
- `data/CoNSeP_point/gt_custom`
- `data/CoNSeP_point/k_func_maps`
- `data/CoNSeP_point/data_splits`
- `data/CoNSeP_point/data_splits/five_fold`

同时保留一套普通 `train/val/test` 划分，并额外生成 5 折交叉验证文件。

In [1]:
from __future__ import annotations

import shutil
from pathlib import Path

import numpy as np

POINT_ROOT = Path("data") / "CoNSeP_point"
TRAIN_ROOT = POINT_ROOT / "CoNSeP_train"
TEST_ROOT = POINT_ROOT / "CoNSeP_test"
MERGED_IMAGE_ROOT = POINT_ROOT / "images"
MERGED_GT_ROOT = POINT_ROOT / "gt_custom"
MERGED_KFUNC_ROOT = POINT_ROOT / "k_func_maps"
SPLIT_ROOT = POINT_ROOT / "data_splits"
FIVE_FOLD_SPLIT_ROOT = SPLIT_ROOT / "five_fold"
LEGACY_SPLIT_ROOT = Path("..") / "data_splits" / "consep"
FOLD_COUNT = 5
RANDOM_SEED = 2026


def _ensure_point_output_dirs() -> None:
    MERGED_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
    MERGED_GT_ROOT.mkdir(parents=True, exist_ok=True)
    MERGED_KFUNC_ROOT.mkdir(parents=True, exist_ok=True)
    SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
    FIVE_FOLD_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)


def _read_split_names(split_path: Path) -> list[str]:
    with split_path.open("r", encoding="utf-8") as handle:
        return [line.strip() for line in handle if line.strip()]


def _copy_if_needed(source_path: Path, destination_path: Path) -> None:
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if not destination_path.exists() or source_path.stat().st_mtime > destination_path.stat().st_mtime:
        shutil.copy2(source_path, destination_path)


def _copy_point_sample(sample_name: str, source_root: Path) -> None:
    image_source = source_root / "images" / sample_name
    gt_source = source_root / "gt_custom" / sample_name.replace(".png", "_gt_dots.npy")
    _copy_if_needed(image_source, MERGED_IMAGE_ROOT / sample_name)
    _copy_if_needed(gt_source, MERGED_GT_ROOT / gt_source.name)

    sample_stem = Path(sample_name).stem
    for kfunc_source in (source_root / "k_func_maps").glob(f"{sample_stem}*"):
        _copy_if_needed(kfunc_source, MERGED_KFUNC_ROOT / kfunc_source.name)


def _write_split_file(split_path: Path, sample_names: list[str]) -> None:
    split_path.parent.mkdir(parents=True, exist_ok=True)
    split_path.write_text("\n".join(sample_names) + "\n", encoding="utf-8")


def _build_five_fold_splits(train_names: list[str], fold_count: int, seed: int) -> None:
    rng = np.random.default_rng(seed)
    shuffled_names = np.asarray(sorted(train_names), dtype=object)
    rng.shuffle(shuffled_names)
    fold_arrays = np.array_split(shuffled_names, fold_count)

    for fold_offset, val_array in enumerate(fold_arrays, start=1):
        val_names = sorted(val_array.tolist())
        train_parts = [fold_arrays[index] for index in range(fold_count) if index != (fold_offset - 1)]
        train_names_fold = sorted(np.concatenate(train_parts).tolist()) if train_parts else []
        _write_split_file(FIVE_FOLD_SPLIT_ROOT / f"fold{fold_offset}_train.txt", train_names_fold)
        _write_split_file(FIVE_FOLD_SPLIT_ROOT / f"fold{fold_offset}_val.txt", val_names)

In [2]:
_ensure_point_output_dirs()

legacy_train_names = _read_split_names(LEGACY_SPLIT_ROOT / "train_split.txt")
legacy_val_names = _read_split_names(LEGACY_SPLIT_ROOT / "val_split.txt")
test_names = sorted(path.name for path in (TEST_ROOT / "images").glob("*.png"))

for sample_name in sorted(set(legacy_train_names + legacy_val_names)):
    _copy_point_sample(sample_name, TRAIN_ROOT)

for sample_name in test_names:
    _copy_point_sample(sample_name, TEST_ROOT)

_write_split_file(SPLIT_ROOT / "train_split.txt", legacy_train_names)
_write_split_file(SPLIT_ROOT / "val_split.txt", legacy_val_names)
_write_split_file(SPLIT_ROOT / "test_split.txt", test_names)
_build_five_fold_splits(legacy_train_names + legacy_val_names, FOLD_COUNT, RANDOM_SEED)

print(f"普通 train 数量: {len(legacy_train_names)}")
print(f"普通 val 数量: {len(legacy_val_names)}")
print(f"独立 test 数量: {len(test_names)}")
print(f"五折 split 输出目录: {FIVE_FOLD_SPLIT_ROOT}")

普通 train 数量: 22
普通 val 数量: 5
独立 test 数量: 14
五折 split 输出目录: data\CoNSeP_point\data_splits\five_fold
